# Feature Families + Binary Target Engineering Matrix

This notebook checks whether Quant Warehouse can combine reusable feature families with binary target-engineering events. It focuses on joinability, target sparsity, and family-level signal separation rather than Rank IC.

Targets covered here:

- FMP event pairs: congress buy/sell, insider buy/sell, analyst upgrades/downgrades, price target raises/cuts, guidance raises/cuts, earnings beats/misses.
- Oracle trade entry labels from stored prices: yearly top-k long/short entries using the batched optimal-trade solver with `YE: 1..12`. Execution prices match `optimal_trader`: buy at high, sell at low, short at low, cover at high.

The goal is to answer: for each feature family, which binary target families have enough joined rows to be worth deeper modeling?

In [1]:
from __future__ import annotations

from pathlib import Path
from time import perf_counter
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'quant_warehouse').exists():
    REPO_ROOT = next(parent for parent in Path.cwd().parents if (parent / 'quant_warehouse').exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
from IPython.display import Markdown, display

from quant_warehouse.platforms.data_providers.fmp.target_engineering.event_pairs import EventPairStore
from quant_warehouse.research_tools import (
    BinaryTargetConfig,
    FamilyEvaluationConfig,
    build_event_target_panel,
    build_fundamental_feature_panel,
    build_oracle_trade_target_panel,
    cap_features_by_quality,
    combine_target_panels,
    evaluate_feature_target_matrix,
    load_fmp_event_pairs,
    screen_fmp_equity_universe,
    summarize_binary_targets,
)
from quant_warehouse.warehouse.api import Warehouse

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 180)

In [2]:
RUN_STARTED = perf_counter()

FEATURE_CONFIG = FamilyEvaluationConfig(
    provider='fmp',
    market_cap_min=100_000_000_000,
    start_date='2018-01-01',
    horizons=(20, 60),
    min_observations=120,
    max_features_per_family=50,
)

TARGET_CONFIG = BinaryTargetConfig(
    provider='fmp',
    start_date=FEATURE_CONFIG.start_date,
    end_date=FEATURE_CONFIG.end_date,
    event_families=(
        'congress',
        'insider',
        'analyst_rating',
        'price_target',
        'guidance',
        'earnings',
    ),
    event_windows=(20, 60),
    oracle_trade_k_by_frequency={'YE': tuple(range(1, 13))},
    oracle_trade_min_profit_pct=0.01,
    oracle_trade_long_entry_price_col='high',  # buy execution
    oracle_trade_long_exit_price_col='low',    # sell execution
    oracle_trade_short_entry_price_col='low',  # short execution
    oracle_trade_short_exit_price_col='high',  # cover execution
)

FEATURE_CONFIG, TARGET_CONFIG

WAREHOUSE = Warehouse()
EVENT_STORE = EventPairStore(
    config=WAREHOUSE.config,
    backend=WAREHOUSE.backend,
    catalog=WAREHOUSE.catalog,
    fundamentals=WAREHOUSE.fundamentals,
    equity_calendar=WAREHOUSE.equity_calendar,
)

## Universe and Feature Families

The notebook starts with FMP equities screened through the OpenBB/FMP screener route, then keeps symbols with the locally stored sections required for the fundamental feature families.

In [3]:
symbols, raw_universe, eligibility, universe_source = screen_fmp_equity_universe(FEATURE_CONFIG, warehouse=WAREHOUSE)

print(f'Universe source: {universe_source}')
print(f'Eligible symbols: {len(symbols)}')
print(symbols[:50])

eligibility.head(20)

Universe source: catalog:fmp:fallback_after_OpenBBError
Eligible symbols: 122
('AAPL', 'ABBV', 'ABT', 'ADI', 'AMAT', 'AMD', 'AMGN', 'AMZN', 'ANET', 'ANTM', 'APH', 'APP', 'ASML', 'AVGO', 'AXP', 'BA', 'BAC', 'BKNG', 'BLK', 'BMY', 'BRK-A', 'BRK-B', 'BX', 'C', 'CAT', 'CDNS', 'COF', 'COP', 'COST', 'CRM', 'CRWD', 'CSCO', 'CVS', 'CVX', 'DE', 'DELL', 'DHR', 'DIS', 'EQIX', 'ETN', 'FTNT', 'GE', 'GEV', 'GILD', 'GLW', 'GOOG', 'GOOGL', 'GS', 'HD', 'HON')


,symbol,eligible,reason,screen_market_cap
0,NVDA,True,ok,4.819979e+12
1,AAPL,True,ok,4.322489e+12
2,GOOGL,True,ok,4.186397e+12
3,GOOG,True,ok,4.185793e+12
4,MSFT,True,ok,2.777787e+12
5,AMZN,True,ok,2.518345e+12
6,SPCX,True,ok,2.020744e+12
7,AVGO,True,ok,1.808594e+12
8,TSLA,True,ok,1.433220e+12
9,META,True,ok,1.427102e+12


In [4]:
raw_feature_panel, raw_feature_metadata, feature_diagnostics, feature_timings = build_fundamental_feature_panel(
    symbols,
    FEATURE_CONFIG,
    warehouse=WAREHOUSE,
)

print({
    'screened_symbols': len(symbols),
    'raw_feature_panel_rows': len(raw_feature_panel),
    'raw_features': raw_feature_metadata['feature'].nunique(),
    **feature_timings,
})

feature_diagnostics.sort_values(['status', 'rows'], ascending=[True, False]).head(20)

{'screened_symbols': 122, 'raw_feature_panel_rows': 247343, 'raw_features': 376, 'raw_panel_build_seconds': 9.878742063883692}


,symbol,status,rows,features,seconds
0,AAPL,ok,2130,376,0.099104
1,ABBV,ok,2130,376,0.070152
2,ABT,ok,2130,376,0.076447
3,ADI,ok,2130,376,0.076548
4,AMAT,ok,2130,376,0.064998
5,AMD,ok,2130,376,0.076841
6,AMGN,ok,2130,376,0.070477
7,AMZN,ok,2130,376,0.076587
8,ANET,ok,2130,376,0.067915
10,APH,ok,2130,376,0.070607


## Binary Event Targets

Event targets are loaded without remote refresh. Cached normalized event-pair data is used when present; missing families are built from already stored FMP historical sections.

In [5]:
events, event_diagnostics, event_load_seconds = load_fmp_event_pairs(
    symbols,
    TARGET_CONFIG,
    event_store=EVENT_STORE,
    include_historical=True,
)

event_symbols = tuple(
    event_diagnostics
    .loc[event_diagnostics['combined_rows'].gt(0), 'symbol']
    .sort_values()
    .tolist()
)
feature_panel = raw_feature_panel.loc[raw_feature_panel['symbol'].isin(event_symbols)].copy()
feature_metadata = raw_feature_metadata.copy()
selected_features, capped_feature_metadata, feature_quality = cap_features_by_quality(
    feature_panel,
    feature_metadata,
    max_features=FEATURE_CONFIG.max_features_per_family,
)
feature_inventory = (
    capped_feature_metadata
    .groupby(['source', 'family'])
    .agg(feature_count=('feature', 'nunique'))
    .reset_index()
    .sort_values(['source', 'family'])
)

event_target_panel, event_target_metadata = build_event_target_panel(
    feature_panel[['symbol', 'date']],
    events,
    TARGET_CONFIG,
)

print({
    'screened_symbols': len(symbols),
    'event_symbols': len(event_symbols),
    'event_rows': len(events),
    'feature_panel_rows_after_event_symbol_filter': len(feature_panel),
    'capped_features': len(selected_features),
    'event_target_rows': len(event_target_panel),
    'event_target_columns': len(event_target_metadata),
    'event_load_seconds': round(event_load_seconds, 3),
})

display(feature_inventory)
event_diagnostics.sort_values('combined_rows', ascending=False).head(20)

{'screened_symbols': 122, 'event_symbols': 119, 'event_rows': 8481, 'feature_panel_rows_after_event_symbol_filter': 244629, 'capped_features': 365, 'event_target_rows': 244629, 'event_target_columns': 36, 'event_load_seconds': 3.363}


,source,family,feature_count
0,financetoolkit,ft_growth_balance,50
1,financetoolkit,ft_growth_cash,50
2,financetoolkit,ft_growth_income,38
3,financetoolkit,ft_ratios_efficiency,5
4,financetoolkit,ft_ratios_liquidity,7
5,financetoolkit,ft_ratios_profitability,14
6,financetoolkit,ft_ratios_solvency,15
7,financetoolkit,ft_ratios_valuation,24
8,fmp,fmp_balance_mcap,50
9,fmp,fmp_cash_mcap,39


,symbol,cached_rows,historical_rows,combined_rows,event_families
73,MSFT,34,945,979,"(analyst_rating, congress, earnings, insider, ..."
0,AAPL,32,820,852,"(analyst_rating, congress, earnings, insider, ..."
78,NVDA,25,799,824,"(analyst_rating, congress, earnings, insider, ..."
7,AMZN,34,719,753,"(analyst_rating, congress, earnings, insider, ..."
46,GOOGL,34,581,615,"(analyst_rating, congress, earnings, insider, ..."
106,TSLA,32,456,488,"(analyst_rating, congress, earnings, insider, ..."
68,META,34,437,471,"(analyst_rating, congress, earnings, insider, ..."
45,GOOG,0,34,34,"(earnings,)"
43,GILD,0,34,34,"(earnings,)"
64,MA,0,34,34,"(earnings,)"


In [ ]:
oracle_target_panel, oracle_target_metadata, oracle_seconds = build_oracle_trade_target_panel(
    event_symbols,
    TARGET_CONFIG,
    warehouse=WAREHOUSE,
)

target_panel = combine_target_panels(event_target_panel, oracle_target_panel)
target_metadata = pd.concat([event_target_metadata, oracle_target_metadata], ignore_index=True)
target_summary = summarize_binary_targets(target_panel, target_metadata)

print({
    'oracle_target_rows': len(oracle_target_panel),
    'oracle_target_columns': len(oracle_target_metadata),
    'oracle_seconds': round(oracle_seconds, 3),
    'combined_target_rows': len(target_panel),
    'combined_target_columns': target_metadata['target'].nunique(),
})

target_summary

## Feature Family x Target Matrix

This matrix asks whether a model row can be formed for each pair. `mean_abs_smd` is a fast family-level separation check: larger values mean the positive and negative target rows look more different on average. It is not a full trading evaluation.

In [7]:
matrix, training_panel = evaluate_feature_target_matrix(
    feature_panel,
    capped_feature_metadata,
    target_panel,
    target_metadata,
    min_rows=120,
    min_positive_rows=10,
    min_feature_coverage=0.5,
)

usable = matrix.query("status == 'usable'").copy()

print({
    'matrix_rows': len(matrix),
    'usable_pairs': len(usable),
    'training_panel_rows': len(training_panel),
    'elapsed_seconds': round(perf_counter() - RUN_STARTED, 3),
})

usable.head(50)

{'matrix_rows': 1080, 'usable_pairs': 975, 'training_panel_rows': 244629, 'elapsed_seconds': 76.83}


,source,feature_family,target_family,target,feature_count,rows,positive_rows,positive_rate,mean_feature_coverage,mean_abs_smd,status
105,financetoolkit,ft_ratios_profitability,event,target_event_next_60d__earnings_beat,14,244629,186133,0.760879,1.000000,0.089184,usable
106,fmp,fmp_income_mcap,event,target_event_next_60d__earnings_beat,31,244629,186133,0.760879,1.000000,0.058788,usable
107,fmp,fmp_daily_mcap_yield,event,target_event_next_60d__earnings_beat,14,244629,186133,0.760879,0.999774,0.046762,usable
108,fmp,fmp_daily_mcap_multiple,event,target_event_next_60d__earnings_beat,14,244591,186133,0.760997,0.998283,0.044572,usable
109,fmp,fmp_cash_mcap,event,target_event_next_60d__earnings_beat,39,244629,186133,0.760879,1.000000,0.043656,usable
110,financetoolkit,ft_ratios_solvency,event,target_event_next_60d__earnings_beat,15,244629,186133,0.760879,1.000000,0.042156,usable
111,fmp,fmp_daily_ev_yield,event,target_event_next_60d__earnings_beat,7,244629,186133,0.760879,1.000000,0.041127,usable
112,financetoolkit,ft_ratios_efficiency,event,target_event_next_60d__earnings_beat,5,244629,186133,0.760879,1.000000,0.040187,usable
113,financetoolkit,ft_ratios_valuation,event,target_event_next_60d__earnings_beat,24,244629,186133,0.760879,0.993944,0.039888,usable
114,fmp,fmp_daily_ev_multiple,event,target_event_next_60d__earnings_beat,7,244591,186133,0.760997,0.999826,0.038407,usable


In [8]:
coverage_pivot = (
    matrix
    .pivot_table(
        index=['source', 'feature_family'],
        columns='target_family',
        values='target',
        aggfunc='count',
        fill_value=0,
    )
    .reset_index()
)

positive_pivot = (
    usable
    .pivot_table(
        index=['source', 'feature_family'],
        columns='target_family',
        values='positive_rows',
        aggfunc='max',
        fill_value=0,
    )
    .reset_index()
)

coverage_pivot

target_family,source,feature_family,event,oracle_trade
0,financetoolkit,ft_growth_balance,36,36
1,financetoolkit,ft_growth_cash,36,36
2,financetoolkit,ft_growth_income,36,36
3,financetoolkit,ft_ratios_efficiency,36,36
4,financetoolkit,ft_ratios_liquidity,36,36
5,financetoolkit,ft_ratios_profitability,36,36
6,financetoolkit,ft_ratios_solvency,36,36
7,financetoolkit,ft_ratios_valuation,36,36
8,fmp,fmp_balance_mcap,36,36
9,fmp,fmp_cash_mcap,36,36


In [ ]:
best_by_target = (
    usable
    .sort_values(['target', 'mean_abs_smd', 'positive_rows'], ascending=[True, False, False])
    .groupby('target')
    .head(3)
    .reset_index(drop=True)
)

best_by_target

In [10]:
sparse_targets = target_summary.query('positive_rows < 10').copy()
usable_targets = usable['target'].nunique() if not usable.empty else 0
usable_feature_families = usable['feature_family'].nunique() if not usable.empty else 0
best_pairs = usable.sort_values('mean_abs_smd', ascending=False).head(10) if not usable.empty else usable

lines = [
    '## Written Analysis',
    '',
    f'- Universe: {len(symbols)} screened FMP symbols with market cap >= ${FEATURE_CONFIG.market_cap_min:,.0f}; {len(event_symbols)} symbols had at least one loaded event pair and were used in the target matrix.',
    f'- Feature side: {capped_feature_metadata["family"].nunique()} capped feature families and {len(selected_features)} selected feature columns.',
    f'- Target side: {target_metadata["target"].nunique()} binary target columns across event and oracle-trade families.',
    f'- Joinability: {len(usable)} feature-family/target pairs are usable under the current thresholds, covering {usable_targets} target columns and {usable_feature_families} feature families.',
]

if not sparse_targets.empty:
    lines.append(f'- Sparse targets: {len(sparse_targets)} target columns have fewer than 10 positive rows and should not be trusted for modeling yet.')

if not best_pairs.empty:
    lines.append('')
    lines.append('Top family/target combinations by fast separation score:')
    for row in best_pairs[['source', 'feature_family', 'target', 'positive_rows', 'mean_abs_smd']].to_dict('records'):
        lines.append(
            f'- {row["source"]}.{row["feature_family"]} -> {row["target"]}: '
            f'{int(row["positive_rows"])} positives, mean_abs_smd={row["mean_abs_smd"]:.4f}'
        )

lines.extend([
    '',
    'Interpretation:',
    '',
    '- A usable pair only means the warehouse can form a training matrix with enough positive binary events. It does not prove tradeability.',
    '- Oracle-trade labels are sparse yearly top-k entry labels from the batched optimal-trade solver; they match the fast `optimal_trader` style better than dense daily horizon labels.',
    '- Congress, insider, guidance, and earnings labels are sparse and should be interpreted at the family level first before actor-specific or event-specific modeling.',
    '- The next practical step is to choose a few usable target families, then add leakage-safe event-date rules and transaction-cost-aware evaluation in Quant Orchestrator.',
])

display(Markdown('\n'.join(lines)))

## Written Analysis

- Universe: 122 screened FMP symbols with market cap >= $100,000,000,000; 119 symbols had at least one loaded event pair and were used in the target matrix.
- Feature side: 15 capped feature families and 365 selected feature columns.
- Target side: 72 binary target columns across event and oracle-trade families.
- Joinability: 975 feature-family/target pairs are usable under the current thresholds, covering 65 target columns and 15 feature families.
- Sparse targets: 7 target columns have fewer than 10 positive rows and should not be trusted for modeling yet.

Top family/target combinations by fast separation score:
- financetoolkit.ft_ratios_profitability -> target_event_on__insider_sell: 534 positives, mean_abs_smd=0.7715
- financetoolkit.ft_ratios_profitability -> target_event_next_20d__analyst_upgrade: 1817 positives, mean_abs_smd=0.6810
- financetoolkit.ft_ratios_profitability -> target_event_on__analyst_upgrade: 214 positives, mean_abs_smd=0.6682
- fmp.fmp_daily_ev_yield -> target_event_next_60d__insider_buy: 240 positives, mean_abs_smd=0.6662
- financetoolkit.ft_ratios_profitability -> target_event_next_60d__analyst_upgrade: 3011 positives, mean_abs_smd=0.6635
- fmp.fmp_daily_ev_yield -> target_event_next_20d__insider_buy: 80 positives, mean_abs_smd=0.6495
- financetoolkit.ft_ratios_profitability -> target_event_on__price_target_raise: 202 positives, mean_abs_smd=0.6089
- financetoolkit.ft_ratios_profitability -> target_event_next_60d__price_target_raise: 2802 positives, mean_abs_smd=0.6078
- financetoolkit.ft_ratios_profitability -> target_event_next_60d__analyst_downgrade: 1881 positives, mean_abs_smd=0.5831
- financetoolkit.ft_ratios_profitability -> target_event_next_20d__price_target_raise: 1884 positives, mean_abs_smd=0.5784

Interpretation:

- A usable pair only means the warehouse can form a training matrix with enough positive binary events. It does not prove tradeability.
- Oracle-trade labels are sparse yearly top-k entry labels from the batched optimal-trade solver; they match the fast `optimal_trader` style better than dense daily horizon labels.
- Congress, insider, guidance, and earnings labels are sparse and should be interpreted at the family level first before actor-specific or event-specific modeling.
- The next practical step is to choose a few usable target families, then add leakage-safe event-date rules and transaction-cost-aware evaluation in Quant Orchestrator.